# Section 1: Setup & Initialization
In this section, we import the necessary libraries for computer vision (`cv2`, `torchvision`), natural language processing (`transformers`), and vector search (`faiss`). We also define our global hyperparameters, embedding sizes, and configure our hardware device (GPU/CPU).

In [ ]:
import numpy as np
import cv2
import pandas as pd
import os
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
import itertools
import torch.optim as optim
import torch.nn.functional as F
import math

from transformers import AutoTokenizer, AutoModel

In [ ]:
!pip install -q faiss-cpu
import faiss

In [ ]:
batch_size = 16
image_size = 224
dropout = 0.1

# Set the environment variable
os.environ["TOKENIZERS_PARALLELISM"] = "true" 

# Get the number of available processors
num_processors = os.cpu_count()

max_length = 200
text_embedding_size = 768  # d_i

shared_embedding_size = 512  # d_e shared embedding space

num_epochs = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Section 2: Data Preprocessing & DataLoaders
Here, we load the Indiana University Chest X-ray dataset. We match the medical images (`projections`) with their corresponding radiological reports (`findings`). 
We then build a custom PyTorch `Dataset` and `DataLoader` to preprocess the images (resize, normalize) and serve batches of `(image, text)` pairs for training.

In [ ]:
df_projections = pd.read_csv('/kaggle/input/chest-xrays-indiana-university/indiana_projections.csv')
df_reports = pd.read_csv('/kaggle/input/chest-xrays-indiana-university/indiana_reports.csv')

In [ ]:
df_reports

In [ ]:
def create_imagescaption():
    images_captions_df = pd.DataFrame({'image': [],
                                        'caption': [],'number_of_words':[]})
    for i in range(len(df_projections)):
        uid = df_projections.iloc[i]['uid']
        image = df_projections.iloc[i]['filename']
        index = df_reports.loc[df_reports['uid'] ==uid]
        
        if not index.empty:    
            index = index.index[0]
            caption = df_reports.iloc[index]['findings']
           
            number_of_words = len(str(caption).split())
    
            if type(caption) == float:
                    continue
            images_captions_df = pd.concat([images_captions_df, pd.DataFrame([{'image': image, 'caption': caption ,'number_of_words':number_of_words}])], ignore_index=True)
    
    images_captions_df["number_of_words"] =  images_captions_df["caption"].apply(lambda text: len(str(text).split()))
    images_captions_df['number_of_words'] = images_captions_df['number_of_words'].astype(int)
    
    return images_captions_df

images_captions_df = create_imagescaption()

In [ ]:
image_filenames = images_captions_df.image.values

train_captions,test_captions =train_test_split(images_captions_df, test_size = 0.2)
print(train_captions.shape)
print (test_captions.shape)

In [ ]:
train_captions

In [ ]:
def preprocess_image(image_path, image_size):
    """
    Loads and preprocesses an image.
    
    Args:
        image_path (str): Path to the image to load.
        target_size (int): The new size of the images (assumes square images).
    
    Returns:
        numpy.ndarray: The preprocessed image.
    """
    # Load the image using OpenCV
    img = cv2.imread(image_path)
    # Resize the image
    img = cv2.resize(img, (image_size, image_size))
    # Convert from BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def display_image(image_np):
    """
    Displays an image.
    
    Args:
        image_np (numpy.ndarray): The image to display.
    """
    plt.imshow(image_np)
    plt.axis('off')  # Hide axis labels
    plt.show()

image = preprocess_image("/kaggle/input/chest-xrays-indiana-university/images/images_normalized/1000_IM-0003-1001.dcm.png",image_size)
display_image (image)
print(image.shape)

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, image_filenames, captions, image_size=image_size):
        """
        Initializes the dataset.
        
        Args:
            image_filenames (list): List of image file paths.
            captions (list): List of corresponding captions.
            image_size (int): Target size for the images.
        """
        self.image_filenames = image_filenames
        self.captions = captions
        self.image_size = image_size
        self.transform = transforms.Compose([transforms.ToTensor()])
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        image_path = f"/kaggle/input/chest-xrays-indiana-university/images/images_normalized/{self.image_filenames[idx]}"
        image = preprocess_image(image_path, self.image_size)
        image = self.transform(image)
        caption = self.captions[idx]
        return image, caption

In [ ]:
# Create dataset instances
train_dataset = ImageTextDataset(train_captions.image.values, train_captions.caption.values) #,image_size)
test_dataset = ImageTextDataset(test_captions.image.values, test_captions.caption.values ) # ,image_size)

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(train_dataset. __getitem__(0)[0].shape)

In [ ]:
images_batch, captions_batch = next(iter(train_dataloader))

for i in range(5):  # Display the first 5 images and captions
    image_to_display = images_batch[i].permute(1, 2, 0).numpy()
    caption_to_display = captions_batch[i]
    display_image(image_to_display)
    print(f"Caption: {caption_to_display}\n\n")

# Section 3: Model Architecture (Encoders & Projections)
To build a multimodal AI, we need to understand both images and text. We use a **Dual-Encoder Architecture**:
1. **Vision Encoder:** A pre-trained ResNet50 model (acting as a stand-in for a ViT) to extract visual features.
2. **Text Encoder:** `Bio_ClinicalBERT`, a specialized model for medical text, to extract linguistic features.
3. **Projection Heads:** Custom neural network layers (`ImageProjection` and `TextProjection`) that map both image and text embeddings into a **shared 512-dimensional embedding space** so they can be directly compared.

In [ ]:
model_name='emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def generate_text_embeddings(texts):
    # Tokenize input texts
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)

    # Generate embeddings
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the embeddings from the last hidden layer
    embeddings = outputs.last_hidden_state[:, 0, :] # <---CLS # outputs.last_hidden_state.mean(dim=1)  # You can use other aggregation methods

    return embeddings

In [ ]:
# List of input texts
texts = [ "Hello", "how", "are", "you"]

# Generate embeddings
embeddings = generate_text_embeddings(texts)
print("Shape of embeddings:", embeddings.shape)  # Should print (num_texts, embedding_size)

In [ ]:
def count_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params

count_trainable_parameters(model)

In [ ]:
class ImageProjection(nn.Module):
    def __init__(self, image_embedding_size, shared_embedding_size):
        super(ImageProjection, self).__init__()
        self.image_projection = nn.Linear(image_embedding_size, shared_embedding_size)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(shared_embedding_size, shared_embedding_size)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(shared_embedding_size)
        
    def forward(self, image_embeddings):
        projected_embeddings = self.image_projection(image_embeddings)
        
        x = self.gelu(projected_embeddings)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected_embeddings
        x = self.layer_norm(x)
        
        return x # projected_embeddings

class TextProjection(nn.Module):
    def __init__(self, text_embedding_size, shared_embedding_size):
        super(TextProjection, self).__init__()
        self.text_projection = nn.Linear(text_embedding_size, shared_embedding_size)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(shared_embedding_size, shared_embedding_size)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(shared_embedding_size)
        
    def forward(self, text_embeddings):
        projected_embeddings = self.text_projection(text_embeddings)
        
        x = self.gelu(projected_embeddings)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected_embeddings
        x = self.layer_norm(x)
        
        return x # projected_embeddings

In [ ]:
# Checking Data Setup
for batch_idx, (images, texts) in enumerate(train_dataloader):
    print( "batch_idx: ", batch_idx, " Image Shape: ", images.shape, "Text Count: ", len(texts) )
    plt.imshow( torch.moveaxis( images[0], 0, 2 ).numpy() )
    plt.show()
    print(texts[0])
    break

# Section 4: Multimodal Alignment (CLIP Loss)
How does the model learn that an X-ray matches its report? We implement **Symmetric Contrastive Learning** (the same math behind OpenAI's CLIP). 
This custom loss function pushes the embeddings of matching image-text pairs closer together in the shared space while pushing apart non-matching pairs.

In [ ]:
class CLIPLoss(nn.Module):

    def __init__(self, init_temperature=0.07):
        super().__init__()

        # log(1 / temperature)
        self.logit_scale = nn.Parameter(
            torch.log(torch.tensor(1.0 / init_temperature)))

    def forward(self,image_projection,text_projection):

        image_projection = F.normalize(image_projection,dim=-1)

        text_projection = F.normalize(text_projection,dim=-1)

        logit_scale = self.logit_scale.exp()

        # Prevent excessively large logits
        logit_scale = torch.clamp(logit_scale,max=100)

        # Pairwise similarities

        logits_per_image = (logit_scale *image_projection @text_projection.T)

        logits_per_text = logits_per_image.T

        # Correct image-text pair is diagonal

        batch_size = image_projection.size(0)

        labels = torch.arange(batch_size,device=image_projection.device)

        # Symmetric contrastive loss

        loss_image = F.cross_entropy(logits_per_image,labels)

        loss_text = F.cross_entropy(logits_per_text,labels)

        loss = (loss_image +loss_text) / 2

        return loss


clip_loss = CLIPLoss(init_temperature=0.07).to(device)

In [ ]:
#  - - - - - - - - - - - - - - - - - -   Create model components   - - - - - - - - - - - - - - - - - -

# - - - - - - - - - - - - - - -- - Image Encodeur Modele  - - - - - - - - - - -- - - - - - - - - -
# Load pre-trained ViT
vit_model = models.vit_b_16(pretrained=True)
# Remove the classification head to get embeddings
vit_model.heads = nn.Identity()
vit_model.to(device)
# Note: ViT outputs shape [B, 768], so update your image_embedding_size accordingly
image_embedding_size = 768

In [ ]:
# - - - - - - - - - - - - - - -- - Text Encodeur Modele = Bio_ClinicalBERT  - - - - - - - - - - -- - - - - - - - - -

model_name='emilyalsentzer/Bio_ClinicalBERT'
text_tokenizer = AutoTokenizer.from_pretrained(model_name)
text_model = AutoModel.from_pretrained(model_name).to(device)
print(" Number of Trainable Parameters in", " Bio_ClinicalBERT model :  ",   count_trainable_parameters(text_model))

In [ ]:
# - - - - - - - - -  Projections  - - - - - - - - -
image_projector = ImageProjection(image_embedding_size, shared_embedding_size).to(device)
print(" Number of Trainable Parameters in", " Image Projection :  ",   count_trainable_parameters(image_projector))
text_projector = TextProjection(text_embedding_size, shared_embedding_size).to(device)
print(" Number of Trainable Parameters in", " Text Projection :  ",   count_trainable_parameters(text_projector))
print("\n - - - - - - - - - - \n \n  Training......  ")

# Section 5: Training Loop & Evaluation
In this section, we train our multimodal model. We use the AdamW optimizer and a learning rate scheduler. 
During evaluation, we test the model's accuracy by seeing if the image embedding's highest cosine similarity matches its actual correct caption from a batch of test data.

In [ ]:
optimizer = optim.AdamW(
    [
        {
            "params": vit_model.parameters(),
            "lr": 1e-5
        },

        {
            "params": text_model.parameters(),
            "lr": 1e-5
        },

        {
            "params": image_projector.parameters(),
            "lr": 1e-4
        },

        {
            "params": text_projector.parameters(),
            "lr": 1e-4
        },

        {
            "params": clip_loss.parameters(),
            "lr": 1e-4
        },
    ],

    weight_decay=1e-4
)

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( optimizer, mode="min", patience=1 , factor=0.8 )

In [ ]:
def train_models_clip():
    history =[]
    # - - - - - - - - -  Training loop  - - - - - - - - -
    for epoch in range(num_epochs):
        start_time = time.time()
        print( " - - - - - - - - - - - Epoch:", epoch+1, " - - - - - - - - - - - - "  )
        vit_model.train()
        text_model.train()
        image_projector.train()
        text_projector.train()
        total_loss = 0.0
        
        for batch_idx, (images, texts) in enumerate(train_dataloader):
            optimizer.zero_grad()
 
            # TEXT
            inputs = tokenizer(texts,return_tensors="pt",padding="max_length",max_length=max_length,truncation=True)
        
            inputs = {key: value.to(device)
                        for key, value in inputs.items()}
        
            outputs = text_model(**inputs)
        
            text_embeddings = (outputs.last_hidden_state[:, 0, :])
        
            text_projection = text_projector(text_embeddings)
        
            # IMAGE
            images = images.to(device)
        
            image_embeddings = vit_model(images)
        
            image_embeddings = image_embeddings.view(image_embeddings.size(0),-1)
        
            image_projection = image_projector(image_embeddings)
        
            # CLIP LOSS
            loss = clip_loss(image_projection,text_projection)
        
            # BACKPROP
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
            if batch_idx % 200 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}/{len(train_dataloader)}], Loss: {total_loss/((batch_idx+1)*batch_size):.4f}")
                
        # - - - - Loss each epoch
        lr_scheduler.step(total_loss)
        end_time = time.time()
        elapsed_time = end_time - start_time
        avg_loss = total_loss / len(train_dataloader)
        history.append(avg_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_loss:.4f}", "  Time Taken: ", elapsed_time, " seconds")
    
    return history

In [ ]:
history = train_models_clip()

In [ ]:
image_projection = image_projector(image_embeddings)

In [ ]:
def save_checkpoint():
    try:
        checkpoint = {
            'vit_model_dict': vit_model.state_dict(),
            'text_model_dict': text_model.state_dict(),
            'image_projector_dict': image_projector.state_dict(),
            'text_projector_dict': text_projector.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }
        torch.save(checkpoint, "/kaggle/working/CLIP_model_from_Scratch_ResNet_DistilBERT1")
        
    except:
        print("Error in some saving")

In [ ]:
save_checkpoint()

In [ ]:
def calculate_accuracy():
    accuracy_counter = 0
    
    vit_model.eval()
    text_model.eval()
    image_projector.eval()
    text_projector.eval()
    
    with torch.no_grad():
        for batch_images, batch_captions in tqdm(test_dataloader):
            batch_size = batch_images.size(0)
            
            # - - - - - - - - -  Forward pass  - - - - - - - - -
            # - - - -  Text  - - - - 
            inputs = tokenizer(batch_captions, return_tensors='pt', padding="max_length", max_length=max_length, truncation=True)
            inputs = inputs.to(device)
            outputs = text_model(**inputs)
            text_embeddings = outputs.last_hidden_state.mean(dim=1)
            text_projection = text_projector(text_embeddings)
            
           # - - - -   image  - - - -  
            images = batch_images.to(device)
            image_embeddings = vit_model(images)
            image_embeddings = image_embeddings.view(image_embeddings.size(0), -1) 
            image_projection = image_projector(image_embeddings)
            
            # Calculate cosine similarities
            for index_text in range(batch_size):
                similarity_scores_list = []
                for index_image in range(len(image_projection)):
                    score = torch.dot( text_projection[index_text], image_projection[index_image] )
                    similarity_scores_list.append( score.cpu().numpy() )
                similarity_scores_list = np.array(similarity_scores_list)
            
                max_index = np.argmax(similarity_scores_list)
                if max_index==index_text:
                    accuracy_counter += 1
    
    total_samples = len(test_dataloader.dataset)
    accuracy = accuracy_counter / total_samples
    print(f"Accuracy (Pencentage of Correct Matching): {accuracy*100:.4f}")
    return accuracy

accuracy = calculate_accuracy()

In [ ]:
def display_image(image_tensor):
    plt.figure()
    plt.imshow(image_tensor.permute(1, 2, 0))
    plt.axis('off')
    plt.show()


def test_trained_models(limits=5): 
    original_captions = []
    predicted_captions = []
    
    vit_model.eval()
    text_model.eval()
    image_projector.eval()
    text_projector.eval()
    
    with torch.no_grad():
        examples_shown = 0
        for batch_images, batch_captions in tqdm(test_dataloader):
            if examples_shown >= limits:
                break
    
            batch_size = batch_images.size(0)

            inputs = tokenizer(batch_captions, return_tensors='pt', padding="max_length", max_length=max_length, truncation=True)
            inputs = inputs.to(device)
            outputs = text_model(**inputs)
            text_embeddings = outputs.last_hidden_state.mean(dim=1)
            text_projection = text_projector(text_embeddings)
             
            images = batch_images.to(device)
            image_embeddings = vit_model(images)
            image_embeddings = image_embeddings.view(image_embeddings.size(0), -1) 
            image_projection = image_projector(image_embeddings)
    
            for index in range(batch_size):
                similarity_scores = torch.matmul(image_projection[index:index+1], text_projection.T).squeeze(0)
                max_index = torch.argmax(similarity_scores).item()
                
                original_caption = batch_captions[index]
                predicted_caption = batch_captions[max_index]
                
                display_image(batch_images[index].cpu())
                print(f"Original Caption: {original_caption}\n")
                print('--------------------------------------')
                print(f"Predicted Caption: {predicted_caption}\n")
                
                examples_shown += 1
                if examples_shown >= limits:
                    break


test_trained_models()

In [ ]:
temperature_value = torch.exp(
    -clip_loss.logit_scale
).item()

print("Trained temperature:", temperature_value)

# Section 6: Explainable AI (XAI) & Visual Grounding
In medical AI, trust is crucial. We need to know *why* a model made a prediction. 
Here, we implement a **Grad-CAM-style Visual Grounding** technique. By attaching forward and backward hooks to the ResNet's spatial feature maps, we trace the gradient of the image-text similarity score back to the pixels. This creates a heatmap showing exactly *where* the model is looking when given a prompt like "pleural effusion."

In [ ]:
def image_text_visual_grounding(image,text,resnet_spatial,image_projector,text_model,text_projector,tokenizer,max_length):
    """
    Generate a text-conditioned visual grounding map
    for a ResNet50 + text encoder CLIP-style model.
    """
    resnet_spatial.eval()
    text_model.eval()
    image_projector.eval()
    text_projector.eval()

    # Image
    image = image.to(device)

    # We need gradients through the spatial feature map.
    feature_maps = resnet_spatial(image)

    # [B, 2048, H, W]
    feature_maps.retain_grad()

    # Global average pooling
    image_embeddings = feature_maps.mean(dim=[2, 3])

    # Projection
    image_projection = image_projector(image_embeddings)

    # Text
    inputs = tokenizer(text,return_tensors="pt",padding="max_length",max_length=max_length,truncation=True)

    inputs = {key: value.to(device)
            for key, value in inputs.items()}

    outputs = text_model(**inputs)

    text_embeddings = outputs.last_hidden_state[:, 0, :]

    text_projection = text_projector(text_embeddings)

    # Normalize embeddings
    image_projection = F.normalize(image_projection,dim=-1)

    text_projection = F.normalize(text_projection,dim=-1)

    # Image-text similarity
    similarity = (image_projection *text_projection).sum(dim=-1)

    # Gradient
    resnet_spatial.zero_grad()
    image_projector.zero_grad()
    text_model.zero_grad()
    text_projector.zero_grad()
    similarity.sum().backward()

    # Gradient × activation
    gradients = feature_maps.grad
    weights = gradients.mean(dim=[2, 3],keepdim=True)

    activation = feature_maps
    cam = (weights * activation).sum(dim=1)
    cam = F.relu(cam)

    # Normalize
    cam_min = cam.flatten(1).min(dim=1)[0]
    cam_max = cam.flatten(1).max(dim=1)[0]

    cam = (cam - cam_min[:, None, None]) / (
        cam_max[:, None, None] - cam_min[:, None, None] + 1e-8)

    return cam.detach(), similarity.detach()

In [ ]:
def generate_visual_grounding(
    image,
    text,
    vit_model,
    text_model,
    image_projector,
    text_projector,
    tokenizer,
    max_length
):

    vit_model.eval()
    text_model.eval()
    image_projector.eval()
    text_projector.eval()

    if isinstance(image, np.ndarray):
        image = torch.from_numpy(image)

    if image.dim() == 3:

        if image.shape[-1] in [1, 3]:
            image = image.permute(2, 0, 1)

        image = image.unsqueeze(0)

    image = image.float().to(device)

    if image.max() > 1:
        image = image / 255.0

    text_inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    input_ids = text_inputs["input_ids"].to(device)
    attention_mask = text_inputs["attention_mask"].to(device)

    captured = {}

    def patch_hook(module, inputs, output):
        captured["patch_embedding"] = output

    hook = vit_model.conv_proj.register_forward_hook(
        patch_hook
    )

    image_output = vit_model(image)

    hook.remove()

    patch_embedding = captured["patch_embedding"]

    print(
        "Patch embedding:",
        patch_embedding.shape
    )

    text_outputs = text_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    text_features = text_outputs.last_hidden_state[:, 0, :]

    image_features = image_projector(
        image_output
    )

    text_features = text_projector(
        text_features
    )

    image_features = F.normalize(
        image_features,
        dim=-1
    )

    text_features = F.normalize(
        text_features,
        dim=-1
    )

    similarity = (
        image_features * text_features
    ).sum(dim=-1)

    print(
        "Similarity:",
        similarity
    )

    gradients = torch.autograd.grad(
        outputs=similarity.sum(),
        inputs=patch_embedding,
        retain_graph=False,
        create_graph=False,
        allow_unused=True
    )[0]

    if gradients is None:

        raise RuntimeError(
            "Gradient with respect to ViT patch embedding "
            "is None. This indicates that the patch embedding "
            "is not directly connected to the similarity "
            "computation in the current ViT graph."
        )

    print(
        "Patch embedding gradients:",
        gradients.shape
    )

    feature_maps = patch_embedding

    weights = gradients.mean(
        dim=(2, 3),
        keepdim=True
    )

    cam = (
        weights * feature_maps
    ).sum(dim=1)

    # 16. ReLU

    cam = F.relu(cam)

    B = cam.shape[0]

    cam_flat = cam.reshape(B, -1)

    cam_min = cam_flat.min(
        dim=1
    )[0].view(B, 1, 1)

    cam_max = cam_flat.max(
        dim=1
    )[0].view(B, 1, 1)

    cam = (
        cam - cam_min
    ) / (
        cam_max - cam_min + 1e-8
    )

    cam = F.interpolate(
        cam.unsqueeze(1),
        size=(
            image.shape[-2],
            image.shape[-1]
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    return (
        cam.detach(),
        similarity.detach(),
        feature_maps.detach()
    )

In [ ]:
text = "pleural effusion"

cam, similarity, feature_maps = generate_visual_grounding(
    image=images,
    text=text,
    vit_model=vit_model,
    text_model=text_model,
    image_projector=image_projector,
    text_projector=text_projector,
    tokenizer=tokenizer,
    max_length=max_length
)

print("Similarity:", similarity)
print("CAM:", cam.shape)
print("Feature maps:", feature_maps.shape)

In [ ]:
def plot_cam_images_similarities_with_text(images,similarity):
    fig, axes = plt.subplots(len(images), 3, figsize=(12, 4 * len(images)))
    
    for i in range(len(images)):
        # Original image
        image_np = images[i].detach().cpu()
    
        # CHW -> HWC
        image_np = image_np.permute(1, 2, 0).numpy()
    
        # normalize for visualization
        image_np = (image_np - image_np.min()) / (
            image_np.max() - image_np.min() + 1e-8 )
    
        axes[i, 0].imshow(image_np)
        axes[i, 0].set_title(f"Image {i+1}")
        axes[i, 0].axis("off")

        # CAM
        axes[i, 1].imshow(cam[i].detach().cpu().numpy(),cmap="jet")
        axes[i, 1].set_title( f"CAM\nSimilarity = {similarity[i].item():.4f}")
        axes[i, 1].axis("off")
    
        # Overlay
        axes[i, 2].imshow(image_np)
        axes[i, 2].imshow(cam[i].detach().cpu().numpy(),cmap="jet",alpha=0.45)
        axes[i, 2].set_title("Pleural Effusion Grounding")
        axes[i, 2].axis("off")
    
    plt.tight_layout()
    plt.show()

plot_cam_images_similarities_with_text(images,similarity)

In [ ]:
with torch.no_grad():

    # -----------------------------------------
    # IMAGE
    # -----------------------------------------
    images_device = images.to(device)

    image_embeddings = vit_model(images_device)

    image_embeddings = image_embeddings.view(
        image_embeddings.size(0),
        -1
    )

    image_projection = image_projector(
        image_embeddings
    )

    # -----------------------------------------
    # TEXT
    # -----------------------------------------
    text = "pleural effusion"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        max_length=max_length,
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    outputs = text_model(**inputs)

    text_embeddings = outputs.last_hidden_state[:, 0, :]

    text_projection = text_projector(
        text_embeddings
    )

    # -----------------------------------------
    # EXACT SCORE USED DURING TRAINING
    # -----------------------------------------

    logits = (
        text_projection @ image_projection.T
    ) / temperature_value


print("Image projection shape:")
print(image_projection.shape)

print("\nText projection shape:")
print(text_projection.shape)

print("\nImage projection norms:")
print(torch.norm(image_projection, dim=1))

print("\nText projection norm:")
print(torch.norm(text_projection, dim=1))

print("\nTraining-style logits:")
print(logits)

# Section 7: Multimodal Retrieval (Foundation for RAG)
In this final section, we build the **Retrieval (R)** component of a RAG pipeline. 
We create a vector database using **FAISS** and populate it with medical documents encoded by our trained Text Encoder. Because our model aligns text and images, we can query this database using text, an image, or a **multimodal query** (image + text combined). The retrieved medical knowledge is then formatted into a context block, ready to be passed to an LLM for final response generation.

In [ ]:
medical_documents = [
    {
        "id": 0,
        "text": """
        Pleural effusion is an abnormal accumulation of fluid
        in the pleural space between the lungs and chest wall.
        On chest radiographs, pleural effusion may appear as
        blunting of the costophrenic angle or as a fluid level.
        """,
        "source": "Medical knowledge - Pleural Effusion"
    },

    {
        "id": 1,
        "text": """
        Pneumonia is an infection of the lung parenchyma.
        Chest radiographs may demonstrate air-space opacity,
        consolidation, or infiltrates. The appearance can vary
        depending on the location and severity of infection.
        """,
        "source": "Medical knowledge - Pneumonia"
    },

    {
        "id": 2,
        "text": """
        Pneumothorax occurs when air accumulates in the pleural
        space. On a chest radiograph, a visible pleural line
        with absence of peripheral lung markings may indicate
        pneumothorax.
        """,
        "source": "Medical knowledge - Pneumothorax"
    },

    {
        "id": 3,
        "text": """
        Cardiomegaly refers to enlargement of the cardiac silhouette.
        On a frontal chest radiograph, an increased cardiothoracic
        ratio can suggest cardiac enlargement.
        """,
        "source": "Medical knowledge - Cardiomegaly"
    },

    {
        "id": 4,
        "text": """
        Pulmonary edema is an abnormal accumulation of fluid in
        the lung interstitium and alveolar spaces. Chest radiographs
        may show bilateral opacities and vascular congestion.
        """,
        "source": "Medical knowledge - Pulmonary Edema"
    }
]

In [ ]:
@torch.no_grad()
def encode_text_for_rag(texts):
    """
    Encode medical documents using the existing
    CLIP-like text encoder + text projector.
    """

    # Tokenize
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(device)

    attention_mask = inputs["attention_mask"].to(device)

    # Text encoder
    text_features = text_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    # ------------------------------------------------
    # IMPORTANT:
    # Adjust this depending on your text_model output.
    # ------------------------------------------------

    if hasattr(text_features, "pooler_output"):
        text_features = text_features.pooler_output

    elif hasattr(text_features, "last_hidden_state"):
        text_features = text_features.last_hidden_state[:, 0]

    # Projection into shared CLIP space
    text_embeddings = text_projector(text_features)

    # Normalize
    text_embeddings = F.normalize(
        text_embeddings,
        dim=-1
    )

    return text_embeddings

In [ ]:
texts = [
    doc["text"]
    for doc in medical_documents
]

knowledge_embeddings = encode_text_for_rag(texts)

In [ ]:
print(knowledge_embeddings.shape)

In [ ]:
knowledge_embeddings_np = (
    knowledge_embeddings
    .detach()
    .cpu()
    .numpy()
    .astype("float32")
)

In [ ]:
embedding_dim = knowledge_embeddings_np.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(knowledge_embeddings_np)

In [ ]:
print("Number of documents:", index.ntotal)

In [ ]:
@torch.no_grad()
def retrieve_documents_from_text(query,top_k=3):

    query_embedding = encode_text_for_rag([query])

    query_embedding = ( query_embedding.detach().cpu().numpy().astype("float32"))

    scores, indices = index.search(query_embedding,top_k)
    
    results = []
    for score, idx in zip(scores[0],indices[0]):
        results.append({
            "score": float(score),
            "text": medical_documents[idx]["text"],
            "source": medical_documents[idx]["source"]
        })

    return results

In [ ]:
results = retrieve_documents_from_text("What is pleural effusion?")

for r in results:
    print("=" * 70)
    print("Score:", r["score"])
    print(r["text"])

In [ ]:
@torch.no_grad()
def encode_image_for_rag(image):

    if isinstance(image, np.ndarray):
        image = torch.from_numpy(image)

    # H,W,C → C,H,W
    if image.ndim == 3 and image.shape[-1] in [1, 3]:
        image = image.permute(2, 0, 1)

    # Add batch dimension
    if image.ndim == 3:
        image = image.unsqueeze(0)

    image = image.float().to(device)

    # If your images are [0,255], convert to [0,1]
    if image.max() > 1:
        image = image / 255.0

    image_features = vit_model(image)

    image_features = image_features.view(
        image_features.size(0),
        -1
    )

    image_embedding = image_projector(
        image_features
    )

    image_embedding = F.normalize(
        image_embedding,
        dim=-1
    )

    return image_embedding

In [ ]:
@torch.no_grad()
def retrieve_documents_from_image(
    image_embedding,
    top_k=3
):

    image_embedding = F.normalize(
        image_embedding,
        dim=-1
    )

    image_embedding_np = (
        image_embedding
        .detach()
        .cpu()
        .numpy()
        .astype("float32")
    )

    scores, indices = index.search(
        image_embedding_np,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({
            "score": float(score),
            "text": medical_documents[idx]["text"],
            "source": medical_documents[idx]["source"]
        })

    return results

In [ ]:
image_embedding = encode_image_for_rag(image)

results = retrieve_documents_from_image(
    image_embedding,
    top_k=3
)

for r in results:

    print("=" * 70)
    print("Score:", r["score"])
    print("Source:", r["source"])
    print(r["text"])

In [ ]:
def create_multimodal_query(
    image_embedding,
    text_embedding,
    image_weight=0.5
):

    text_weight = 1.0 - image_weight

    query_embedding = (
        image_weight * image_embedding
        +
        text_weight * text_embedding
    )

    query_embedding = F.normalize(
        query_embedding,
        dim=-1
    )

    return query_embedding

In [ ]:
image_embedding = encode_image_for_rag(image)

text_embedding = encode_text_for_rag(
    ["What abnormality is visible in this chest X-ray?"]
)

query_embedding = create_multimodal_query(
    image_embedding,
    text_embedding,
    image_weight=0.5
)

In [ ]:
query_embedding_np = (
    query_embedding
    .detach()
    .cpu()
    .numpy()
    .astype("float32")
)

scores, indices = index.search(
    query_embedding_np,
    3
)

In [ ]:
for score, idx in zip(
    scores[0],
    indices[0]
):

    print("=" * 70)
    print("Score:", score)
    print("Source:", medical_documents[idx]["source"])
    print(medical_documents[idx]["text"])

In [ ]:
def build_rag_context(results):

    context = ""

    for i, result in enumerate(results):

        context += f"""
Evidence {i+1}
Source: {result['source']}

{result['text']}

"""

    return context

In [ ]:
context = build_rag_context(results)

print(context)

In [ ]:
from transformers import pipeline

# Load a small generative LLM (e.g., Llama-3, Phi-3, or DistilGPT2 for testing)
generator = pipeline('text-generation', model='distilgpt2')

In [ ]:
user_question = "What abnormality is visible in this chest X-ray?"
prompt = f"Context: {context}\n\nQuestion: {user_question}\n\nAnswer:"

response = generator(prompt, max_new_tokens=50)
print(response[0]['generated_text'])

# Section 8: Conclusion and Future Directions

In this notebook, we successfully built an end-to-end multimodal AI pipeline for medical imaging. By bridging a Vision Transformer (ViT) and Bio_ClinicalBERT into a shared latent space via contrastive learning, the model learned to semantically align chest X-rays with radiological reports. Furthermore, we extended this baseline by introducing Explainable AI (XAI) for visual grounding and a Retrieval-Augmented Generation (RAG) framework to provide clinically contextualized answers.

While this serves as a robust proof-of-concept, real-world clinical deployment requires scaling both the data and the architecture. As part of future research, I plan to address the following areas:

1. **Scaling Contrastive Learning:** The current implementation uses a batch size of 16 due to hardware constraints. Because InfoNCE/CLIP loss relies heavily on large batch sizes for negative sampling, future iterations will utilize Gradient Caching or Momentum Contrast (MoCo) to simulate larger batches and improve embedding separation.
2. **Advanced Transformer Interpretability:** The current XAI implementation hooks into the ViT's initial projection layer (`conv_proj`). While this successfully extracts gradients, future work will implement *Attention Rollout* or *Transformer Interpretability Beyond Attention Visualization* (Chefer et al.) on the deeper self-attention blocks to capture better global contextual grounding.
3. **Upgrading the Generative LLM:** The DistilGPT2 model used in the final RAG pipeline was chosen for its lightweight footprint. I plan to swap this with a modern, domain-specific open-weights model, such as BioMistral or Llama-3 (8B), to prevent medical hallucinations and improve reasoning.
4. **Expanding the Vector Database:** The FAISS index currently acts as a micro-database. Future steps include encoding the entirety of the Indiana University (or MIMIC-CXR) historical reports corpus. This will allow the RAG system to retrieve the most similar historical patient cases based on a novel multimodal query (Image + Text), providing doctors with powerful, evidence-based reference cases.